In [7]:
%reset -f
from IPython import get_ipython
import pandas as pd
import gc
import os
import numpy as np
import seaborn as sns
from collections import Counter
import subprocess
gc.collect()
import warnings
warnings.filterwarnings('ignore')
stata_exe = "C:/Program Files/Stata17/StataMP-64.exe"

In [ ]:
os.chdir('G:\Kuangyu_Temp\Outsource')
df1 = pd.read_stata('lenth15.dta')
df1['product_id'] = df1['product_id'].astype(str)
df1['firm_id'] = df1['firm_id'].astype(str)
df2 = df1[df1['is_output']==1]#产出品
df3 = df1[df1['is_output']==0]#投入品

df_output = df2
df_input = df3

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)

print(df3['product_id'].nunique())
print(df2['product_id'].nunique())
print(df1['product_id'].nunique())

# Step1：去掉1，3位的
df2['product_id_9'] = df2['product_id'].str[:9]
df3['product_id_9'] = df3['product_id'].str[:9]

def is_high_level(code):
    return(
        code[1:] == '0'*(len(code)-1) or
        code[3:] == '0'*(len(code)-3)
    )
    
df_high = df2[df2['product_id'].apply(is_high_level)]
df_low = df2[~df2['product_id'].apply(is_high_level)]

print('product_id')
print(df2['product_id'].nunique())
print("high level",df_high['product_id'].nunique())
print("low level",df_low['product_id'].nunique())

print('v')
print(df2['v'].sum()/100000000)
print(df_high['v'].sum()/100000000)
print(df_low['v'].sum()/100000000)

print('firm_id')
print(df2['firm_id'].nunique())
print(df_high['firm_id'].nunique())
print(df_low['firm_id'].nunique())


# Step2： 筛选5、7位
df_low_group = df_low.groupby('product_id')['v'].sum().reset_index()

df_low_group['product_id'] = df_low_group['product_id'].astype(str)
df_low_group['product_id_5'] = df_low_group['product_id'].str[:5]
df_low_group['product_id_7'] = df_low_group['product_id'].str[:7]
df_low_group['product_id_9'] = df_low_group['product_id'].str[:9]


#截取前七位码，如果没有重复的，那么说明该产品至多到7位
counts_1 = df_low_group.groupby('product_id_7')['product_id_9'].nunique()
single_prefixes_1 = counts_1[counts_1 == 1].index
#同理，梳理五位码
counts_2 = df_low_group.groupby('product_id_5')['product_id_7'].nunique()
single_prefixes_2 = counts_2[counts_2 == 1].index

filtered_df_1 = df_low_group[df_low_group['product_id_7'].isin(single_prefixes_1)]
filtered_df_2 = df_low_group[df_low_group['product_id_5'].isin(single_prefixes_2)]


#查询各位码中有多少个重复的
five_in_seven = [item for item in single_prefixes_1 if item.endswith('00')]#这里包含了全部96个的5位码
seven_in_seven = [item for item in single_prefixes_1 if not item.endswith('00')]
#把筛选出7位码的
print(len(seven_in_seven))
for i in five_in_seven:
    if i[:-2] in single_prefixes_2:
        seven_in_seven.append(i)
print(len(five_in_seven))
print(len(seven_in_seven))

seven_in_seven = [i + '00' for i in seven_in_seven]

#把其他的9位码整出来
product_id = df_low_group['product_id_9'].drop_duplicates().tolist()
product_id_9 = [item for item in product_id if not item.endswith('00')]

print(len(product_id_9))

product_id_9_final = product_id_9+seven_in_seven
print(len(product_id_9_final))

# Step3： 完成9位码的筛选
df_output = df2[df2['product_id_9'].isin(product_id_9_final)]
print(df_output['product_id'].nunique())
df_output = df_output.drop(columns={'product_id'})
df_output = df_output.rename(columns={'product_id_9':'product_id'})
df_output = df_output.groupby(['firm_id', 'product_id', 'is_output','year'])['v'].sum().reset_index()

df_input = df3[df3['product_id_9'].isin(product_id_9_final)]
print(df_input['product_id'].nunique())
df_input= df_input.drop(columns={'product_id'})
df_input = df_input.rename(columns={'product_id_9':'product_id'})
df_input = df_input.groupby(['firm_id', 'product_id', 'is_output','year'])['v'].sum().reset_index()

output_firm_list = df_output['firm_id'].drop_duplicates().tolist()
input_firm_list = df_input['firm_id'].drop_duplicates().tolist()
firm_list = set(output_firm_list)&set(input_firm_list)

print('firm')
print(df_input['firm_id'].nunique())
print(df_output['firm_id'].nunique())
df_input['firm_id'] = df_input['firm_id'].astype(str)
df_output['firm_id'] = df_output['firm_id'].astype(str)

print("output_value",df_output['v'].sum()/100000000)
print("input_value", df_input['v'].sum()/100000000)
print("all_value", df_output['v'].sum()/100000000+df_input['v'].sum()/100000000)

df_output = df_output[df_output['firm_id'].isin(firm_list)]
df_input = df_input[df_input['firm_id'].isin(firm_list)]
df1 = pd.concat([df_input, df_output])

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)
df1.to_stata('lenth9.dta', write_index=False)

产出表
Firm Count 7778255
Product_count 4058
Value 4257560.431760255
投入表
Firm Count 7243728
Product_count 4058
Value 4109195.884376819
4058
4058
4058
product_id
4058
high level 47
low level 4011
v
4257560.431760255
27767.91418539248
4229792.5175748635
firm_id
7778255
1084854
7741138
191
96
214
2564
2778
3469
3469
firm
7228861
7732946
output_value 4210862.867213221
input_value 4064494.9975650487
all_value 8275357.86477827
产出表
Firm Count 7191877
Product_count 2778
Value 4192566.5938059567
投入表
Firm Count 7191877
Product_count 2778
Value 4062909.7821717137


In [4]:
df_output = df_output[df_output['firm_id'].isin(firm_list)]
df_input = df_input[df_input['firm_id'].isin(firm_list)]
df1 = pd.concat([df_input, df_output])

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)

产出表
Firm Count 7191162
Product_count 2778
Value 4191709.8434823374
投入表
Firm Count 7191162
Product_count 2778
Value 4061756.542314181


In [ ]:
agg = (
    df1
    .groupby(['firm_id', 'product_id', 'is_output'], as_index = False)['v']
    .sum()
)

wide = (
    agg
    .pivot_table(index = ['firm_id', 'product_id'],
                    columns = 'is_output',
                    values = 'v',
                    aggfunc = 'sum',
                    fill_value = 0)
    .rename(columns = {0:'v_input', 1:'v_output'})
    .reset_index()
)

wide['diff'] = wide['v_output'] - wide['v_input']

mask_pos = wide['diff'] > 0
mask_neg = wide['diff'] < 0

res_out = wide.loc[mask_pos, ['firm_id', 'product_id']].copy()
res_out['is_output'] = 1
res_out['input_output'] = 'output'
res_out['v'] = wide.loc[mask_pos, 'diff'].values

res_in = wide.loc[mask_neg, ['firm_id', 'product_id']].copy()
res_in['is_output'] = 0
res_in['input_output'] = 'input'
res_in['v'] = (-wide.loc[mask_neg, 'diff']).values

df_hedged = pd.concat([res_in, res_out], ignore_index=True)

df_output = res_out
df_input = res_in

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)
df_hedged.to_stata('lenth9_clean.dta', write_index=False)

产出表
Firm Count 6962763
Product_count 2778
Value 2614453.059125898
投入表
Firm Count 7150934
Product_count 2778
Value 2484499.7579577425
